In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [4]:
X_train = pd.read_csv('../data/processed/X_train_smote.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train_smote.csv')
y_test = pd.read_csv('../data/processed/y_test.csv')

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train.drop(columns=['Satisfaction Score'], inplace=True)
X_test.drop(columns=['Satisfaction Score'], inplace=True)
X_train_nn = scaler.fit_transform(X_train)
X_test_nn = scaler.transform(X_test)

In [6]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    Multiply,
    Softmax,
    BatchNormalization,
    Activation
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)
from tensorflow.keras.optimizers import AdamW
from sklearn.metrics import f1_score

In [7]:
def build_attention_model(
    input_dim,
    hidden_units,
    dropout_rate,
    learning_rate,
    l2_lambda=1e-4
):

    inputs = Input(shape=(input_dim,))

    # Feature Attention
    attention = Dense( 128,
                       kernel_regularizer=l2(l2_lambda)
                       )(inputs)
    attention = BatchNormalization()(attention)
    attention = Activation('tanh')(attention)
    
    attention = Dense(
          input_dim,
          activation="softmax"
    )(attention)
    
    weighted = Multiply()([inputs, attention])

    x = Dense(hidden_units,kernel_regularizer=l2(l2_lambda)
              )(weighted)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Dropout(dropout_rate)(x)

    x = Dense(
               hidden_units // 2,
               kernel_regularizer=l2(l2_lambda)
     )(x)

    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Dropout(dropout_rate)(x)

    outputs = Dense(1, activation="sigmoid")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    Optimizer = AdamW(learning_rate=learning_rate, weight_decay=l2_lambda)

    model.compile(
        optimizer=Optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy",
                 tf.keras.metrics.AUC(name='auc')]
    )

    return model

In [8]:
def objective_attention(trial):

    hidden_units = trial.suggest_categorical(
        "hidden_units",
        [32, 64, 128]
    )

    dropout = trial.suggest_float(
        "dropout",
        0.2,
        0.5
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        1e-2,
        log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [16, 32, 64]
    )

    model = build_attention_model(
        input_dim=X_train_nn.shape[1],
        hidden_units=hidden_units,
        dropout_rate=dropout,
        learning_rate=learning_rate
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )

    history = model.fit(
        X_train_nn,
        y_train,
        validation_split=0.2,
        epochs=100,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=0
    )

    y_prob = model.predict(X_test_nn, verbose=0).flatten()

    y_pred = (y_prob >= 0.5).astype(int)

    return f1_score(y_test, y_pred)

In [9]:

import optuna


study = optuna.create_study(
    direction="maximize",
    study_name="Attention_NN"
)

study.optimize(
    objective_attention,
    n_trials=30,
    show_progress_bar=True
)

[I 2026-07-31 16:27:28,675] A new study created in memory with name: Attention_NN


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-07-31 16:28:11,236] Trial 0 finished with value: 0.6694444444444444 and parameters: {'hidden_units': 32, 'dropout': 0.44358068755617586, 'learning_rate': 0.008232776482212337, 'batch_size': 16}. Best is trial 0 with value: 0.6694444444444444.
[I 2026-07-31 16:28:36,432] Trial 1 finished with value: 0.6426229508196721 and parameters: {'hidden_units': 128, 'dropout': 0.3220215473522223, 'learning_rate': 0.008968134145884004, 'batch_size': 16}. Best is trial 0 with value: 0.6694444444444444.
[I 2026-07-31 16:29:06,457] Trial 2 finished with value: 0.6869455006337135 and parameters: {'hidden_units': 32, 'dropout': 0.23157806819312693, 'learning_rate': 0.004037688696973503, 'batch_size': 16}. Best is trial 2 with value: 0.6869455006337135.
[I 2026-07-31 16:29:29,535] Trial 3 finished with value: 0.6833541927409261 and parameters: {'hidden_units': 128, 'dropout': 0.49697273808017595, 'learning_rate': 0.000545892887974907, 'batch_size': 32}. Best is trial 2 with value: 0.6869455006337

In [10]:
print(study.best_params)
print(study.best_value)

{'hidden_units': 32, 'dropout': 0.4060536808137462, 'learning_rate': 0.0004123316546587999, 'batch_size': 32}
0.7210526315789474


In [11]:
params = study.best_params

attention_model = build_attention_model(
    input_dim=X_train_nn.shape[1],
    hidden_units=params["hidden_units"],
    dropout_rate=params["dropout"],
    learning_rate=params["learning_rate"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
)

attention_model.fit(
    X_train_nn,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=params["batch_size"],
    callbacks=[early_stop,reduce_lr],
    verbose=1
)

Epoch 1/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6489 - auc: 0.6886 - loss: 0.6428 - val_accuracy: 0.7784 - val_auc: 0.0000e+00 - val_loss: 0.6872 - learning_rate: 4.1233e-04
Epoch 2/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7448 - auc: 0.8133 - loss: 0.5293 - val_accuracy: 0.9595 - val_auc: 0.0000e+00 - val_loss: 0.5048 - learning_rate: 4.1233e-04
Epoch 3/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7896 - auc: 0.8554 - loss: 0.4799 - val_accuracy: 0.9632 - val_auc: 0.0000e+00 - val_loss: 0.3732 - learning_rate: 4.1233e-04
Epoch 4/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8135 - auc: 0.8794 - loss: 0.4420 - val_accuracy: 0.9734 - val_auc: 0.0000e+00 - val_loss: 0.2885 - learning_rate: 4.1233e-04
Epoch 5/100
207/207 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8309 - auc: 0.8961 - loss: 0.4139 - val_accuracy: 0.9704 - val_auc: 0.0000e+00 - val_loss: 0.2355 - learning_rate: 4.1233e-04
Epoch 6/100
207/207 ━━━━━━━━━━━━━━━

In [12]:
from sklearn.metrics import classification_report, confusion_matrix


y_prob = attention_model.predict(X_test_nn).flatten()
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
              precision    recall  f1-score   support

           0       0.91      0.86      0.88      1035
           1       0.67      0.75      0.71       374

    accuracy                           0.83      1409
   macro avg       0.79      0.81      0.80      1409
weighted avg       0.84      0.83      0.84      1409

[[893 142]
 [ 92 282]]


In [13]:
import os
import joblib

# Create Project/models/test1
os.makedirs("../models/test1", exist_ok=True)

# Save models
joblib.dump(attention_model, "../models/test1/attention_model.pkl")

['../models/test1/attention_model.pkl']